# 📄 Document-Level OCR — Colab Training (H100 AUTOPILOT, resume-safe)

Fine-tunes the **ByT5 post-OCR corrector**, evaluates the **CER/WER reduction** vs the identity +
SymSpell baselines, runs the **agent** on a document, and generates `report.pdf` + `slides.pptx`.

**Auto-adapts** to H100 / A100 / L4 / T4 (batch + precision; T4 → fp16 + `t5-small`) and
**resumes** from the last checkpoint on Drive. Set the Controls and `Runtime → Run all`.


In [ ]:
#@title 0) Controls — set these, then `Runtime → Run all`  { display-mode: "form" }
GIT_REPO_URL = "https://github.com/ledinhminhquan/07_Document_Level_OCR" #@param {type:"string"}BRANCH = "main" #@param {type:"string"}
PROJECT_DIR_NAME = "07_Document_Level_OCR" #@param {type:"string"}
USE_DRIVE = True #@param {type:"boolean"}
DRIVE_SUBDIR = "dococr" #@param {type:"string"}
BASE_MODEL = "auto" #@param ["auto", "google/byt5-small", "google-t5/t5-small"]
USE_REAL = True #@param {type:"boolean"}
TRAIN_SIZE = 60000 #@param {type:"integer"}
EPOCHS = 3 #@param {type:"integer"}
print("Controls set. Repo:", GIT_REPO_URL or "(will copy from Drive)")

In [ ]:
#@title 1) Check the GPU
!nvidia-smi -L || echo "No GPU — Runtime > Change runtime type > GPU"

In [ ]:
#@title 2) Mount Drive + artifact paths & HF caches  (BEFORE importing torch)
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ART = f"/content/drive/MyDrive/{DRIVE_SUBDIR}"
else:
    ART = "/content/dococr"
os.makedirs(ART, exist_ok=True)
os.environ["DOCOCR_ARTIFACTS_DIR"] = ART
os.environ["DOCOCR_MODEL_DIR"] = f"{ART}/models"
os.environ["DOCOCR_DATA_DIR"]  = f"{ART}/data"
os.environ["DOCOCR_RUN_DIR"]   = f"{ART}/runs"
os.environ["DOCOCR_OUTPUT_DIR"] = f"{ART}/outputs"
os.environ["HF_HOME"] = f"{ART}/hf_cache"
print("Artifacts (Drive-persisted) ->", ART)

In [ ]:
#@title 3) System packages (Tesseract OCR + poppler + OpenCV libs)
!apt-get -qq install -y tesseract-ocr tesseract-ocr-eng poppler-utils libgl1 libglib2.0-0 > /dev/null 2>&1 && echo "system deps ready"

In [ ]:
#@title 4) Get the project source (git clone, or copy from Drive)
import os
WORK = "/content/" + PROJECT_DIR_NAME
if GIT_REPO_URL:
    if not os.path.exists(WORK):
        rc = os.system(f'git clone -b "{BRANCH}" "{GIT_REPO_URL}" "{WORK}"')
        if rc != 0:
            os.system(f'git clone "{GIT_REPO_URL}" "{WORK}"')
else:
    src = f"/content/drive/MyDrive/{PROJECT_DIR_NAME}"
    assert os.path.exists(src), f"Set GIT_REPO_URL or upload the project to Drive/{PROJECT_DIR_NAME}"
    os.system(f'cp -r "{src}" "{WORK}"')
os.chdir(WORK)
print("cwd:", os.getcwd())

In [ ]:
#@title 5) Install dependencies (Colab-safe: NEVER reinstall torch)
%cd $WORK
!pip -q install -r requirements_colab.txt
!pip -q install -e . --no-deps
print("Dependencies installed (torch untouched).")

In [ ]:
#@title 6) Verify environment + performance knobs (TF32)
import torch, transformers
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print("torch", torch.__version__, "| transformers", transformers.__version__)
print("GPU:", GPU, "| CUDA:", torch.cuda.is_available())

In [ ]:
#@title 7) Auto GPU profile (batch + precision; effective batch held at 256)
import torch
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
EFF = 256
prof = {"base_model": "google/byt5-small"}
if "H100" in GPU:   bs, bf16, fp16, ckpt = 32, True,  False, False
elif "A100" in GPU: bs, bf16, fp16, ckpt = 16, True,  False, True
elif "L4" in GPU:   bs, bf16, fp16, ckpt = 8,  True,  False, True
elif "T4" in GPU:   bs, bf16, fp16, ckpt = 4,  False, True,  True; prof["base_model"] = "google-t5/t5-small"
else:               bs, bf16, fp16, ckpt = 2,  False, False, True; prof["base_model"] = "google-t5/t5-small"
if BASE_MODEL != "auto":
    prof["base_model"] = BASE_MODEL
accum = max(1, EFF // bs)
prof.update(per_device_train_batch_size=bs, gradient_accumulation_steps=accum, bf16=bf16, fp16=fp16, gradient_checkpointing=ckpt)
print(GPU, "->", prof)

In [ ]:
#@title 8) Write the Colab training config  (configs/train_colab.yaml)
import yaml
lr = 5.0e-4 if "byt5" in prof["base_model"] else 3.0e-4
cfg = {
    "data": {"synthetic_train_size": int(TRAIN_SIZE), "synthetic_val_size": 4000,
             "synthetic_test_size": 4000, "use_real": bool(USE_REAL), "seed": 42},
    "model": {**prof, "num_train_epochs": int(EPOCHS), "learning_rate": lr,
              "per_device_eval_batch_size": 64, "tf32": True, "group_by_length": True,
              "label_smoothing_factor": 0.1, "early_stopping_patience": 4, "eval_steps": 500, "save_steps": 500},
}
open("configs/train_colab.yaml", "w").write(yaml.safe_dump(cfg, sort_keys=False))
print(open("configs/train_colab.yaml").read())

In [ ]:
#@title 9) Build the corpus + preview synthetic (noisy -> clean) pairs
!dococr --config configs/train_colab.yaml data --task corpus
from dococr.config import load_config
from dococr.data.ocr_noise import OCRNoiseGenerator
for e in OCRNoiseGenerator(load_config("configs/train_colab.yaml").data).generate(3, seed=0):
    print("NOISY:", e["noisy"][:90])
    print("CLEAN:", e["clean"][:90], "\n")

## ⭐ ONE BUTTON — autopilot (resume-safe)
Trains the corrector → evaluates (vs baselines) → analysis → `report.pdf` + `slides.pptx`.
**Re-run after a disconnect to RESUME** from the last checkpoint on Drive.

In [ ]:
#@title 10) ⭐ ONE BUTTON autopilot  (re-run to resume)
!dococr --config configs/train_colab.yaml autopilot

## Individual steps (optional) — idempotent + resume-safe

In [ ]:
#@title 11a) Train only (resumes from the last checkpoint)
!dococr --config configs/train_colab.yaml train

In [ ]:
#@title 11b) Evaluate — corrector vs identity/dictionary (CER/WER reduction)
!dococr evaluate --which test
!dococr evaluate --which real

In [ ]:
#@title 12) Diagnostics: eval metrics + model metadata
import json, os
rd = os.environ["DOCOCR_RUN_DIR"]
try:
    ev = json.load(open(f"{rd}/eval/latest.json"))
    print("SUMMARY:", json.dumps(ev.get("summary", {}), indent=2))
    print("MODEL:", json.dumps(ev.get("model", {}), indent=2))
except Exception as e:
    print("run evaluate first:", e)

## ✅ Test the trained model

In [ ]:
#@title 13a) Post-OCR correction: trained model vs identity (tricky OCR text)
from dococr.config import load_config
from dococr.models.corrector import load_corrector
cfg = load_config("configs/train_colab.yaml")
corr = load_corrector(cfg.model, prefer="neural")
tests = [
    "The cornpany reported steacly growth in the thlrd quarter of 2O21.",
    "Revenue lncreased by l2% compared t0 the previous year.",
    "0perating c0sts fell by nearly a thlrd after the new pollcy.",
]
for t in tests:
    print("OCR :", t)
    print("FIX :", corr.correct(t), "\n")

In [ ]:
#@title 13b) Full agent on a real rendered document page (OCR + structure)
from dococr.agent.doc_agent import DocumentAgent
from dococr.data.samples import render_page
agent = DocumentAgent(load_config("configs/train_colab.yaml"), load_model=True)
job = agent.process(image=render_page(), filename="page.png", save=False)
sd = job.to_dict()
print("status:", sd["status"], "| blocks:", sd["n_blocks"], "| ocr:", sd["model_versions"].get("ocr_engine"))
print("decisions:", [(d["id"], d["branch"]) for d in sd["decisions"]])
print("\n--- MARKDOWN ---\n" + sd["markdown"][:600])

In [ ]:
#@title 14) Locate deliverables (report.pdf + slides.pptx + bundle)
import glob, os
sub = sorted(glob.glob(os.environ["DOCOCR_ARTIFACTS_DIR"] + "/submission/*"))
if sub:
    print("Submission bundle ->", sub[-1])
    for f in sorted(glob.glob(sub[-1] + "/*")):
        print("  ", os.path.basename(f), os.path.getsize(f), "bytes")
else:
    print("Run the autopilot cell (10) to produce report.pdf + slides.pptx.")

In [ ]:
#@title 15) (Optional) Serve the API + Gradio UI
# !dococr serve --ui --port 7860
print("Uncomment the line above to launch the FastAPI + Gradio app.")

## ✅ Final checklist
- [ ] GPU profile picked (cell 7) — H100/A100 → bf16, T4 → fp16 + t5-small
- [ ] Corpus built (cell 9), training finished (cell 10 / 11a)
- [ ] `evaluate` shows the corrector **reduces CER** vs identity, with low % degraded (cell 11b / 12)
- [ ] The trained model fixes the tricky OCR text (cell 13a)
- [ ] The agent OCRs + structures a real page (cell 13b)
- [ ] `report.pdf` + `slides.pptx` produced (cell 14)
- [ ] Everything persisted to Drive (re-run cell 10 to resume after a disconnect)
